In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep
from IPython.display import display


file_path = "results/data.csv"
df = pd.read_csv(file_path, comment="#")


df = df[df["AB Group"].isin(["A", "B"])].copy()
df["Active users"] = pd.to_numeric(df["Active users"], errors="coerce")


def get_active_users(step_name, group_name):
    row = df[(df["Step"] == step_name) & (df["AB Group"] == group_name)]
    if row.empty:
        raise ValueError(f"Missing data for step={step_name}, group={group_name}")
    return int(row["Active users"].iloc[0])

A_start = get_active_users("1. start", "A")
B_start = get_active_users("1. start", "B")

A_upload = get_active_users("2. upload", "A")
B_upload = get_active_users("2. upload", "B")

A_viz = get_active_users("3. visualization", "A")
B_viz = get_active_users("3. visualization", "B")


counts_df = pd.DataFrame({
    "Stage": ["Start", "Upload", "Visualization"],
    "Group A": [A_start, A_upload, A_viz],
    "Group B": [B_start, B_upload, B_viz],
    "Total": [A_start + B_start, A_upload + B_upload, A_viz + B_viz]
})


def stage_test(stage_name, success_A, total_A, success_B, total_B):
    # proportions
    pA = success_A / total_A
    pB = success_B / total_B
    
    # z-test
    z_stat, p_value = proportions_ztest(
        np.array([success_A, success_B]),
        np.array([total_A, total_B])
    )
    
    # effect size
    effect = pA - pB
    
    # CI
    ci_low, ci_high = confint_proportions_2indep(
        success_A, total_A,
        success_B, total_B,
        method="wald"
    )
    
    # decision
    decision = "Reject H0" if p_value < 0.05 else "Fail to reject H0"
    
    return {
        "Stage": stage_name,
        "A Success / Total": f"{success_A}/{total_A}",
        "B Success / Total": f"{success_B}/{total_B}",
        "A Rate": round(pA, 4),
        "B Rate": round(pB, 4),
        "Effect (A-B)": round(effect, 4),
        "95% CI": f"({ci_low:.4f}, {ci_high:.4f})",
        "Z-stat": round(z_stat, 4),
        "P-value": round(p_value, 4),
        "Decision": decision
    }


results = []

# Stage 1: start -> upload
results.append(stage_test(
    "Start → Upload",
    success_A=A_upload, total_A=A_start,
    success_B=B_upload, total_B=B_start
))

# Stage 2: upload -> visualization
results.append(stage_test(
    "Upload → Create Plot",
    success_A=A_viz, total_A=A_upload,
    success_B=B_viz, total_B=B_upload
))

results_df = pd.DataFrame(results)


print("Stage Counts")
display(counts_df)

print("\nStatistical Results")
display(results_df)



Stage Counts


,Stage,Group A,Group B,Total
0,Start,75,85,160
1,Upload,51,42,93
2,Visualization,32,36,68



Statistical Results


,Stage,A Success / Total,B Success / Total,A Rate,B Rate,Effect (A-B),95% CI,Z-stat,P-value,Decision
0,Start → Upload,51/75,42/85,0.6800,0.4941,0.1859,"(0.0361, 0.3357)",2.3783,0.0174,Reject H0
1,Upload → Create Plot,32/51,36/42,0.6275,0.8571,-0.2297,"(-0.3994, -0.0600)",-2.4864,0.0129,Reject H0
